# Anonymization Region Visualization

Visualize the rectangular facial anonymization mask.
The mask covers the full face width (including ears) from above Nz down to chin/neck.

In [1]:
import numpy as np
import pyvista as pv

import cedalion
import cedalion.io
import cedalion.plots
from cedalion import units
from cedalion.geometry.photogrammetry.anonymization import (
    detect_landmarks_from_nasion,
    detect_facial_landmarks,
    get_facial_region_mask,
)
from cedalion.dataclasses import VTKSurface

pv.set_jupyter_backend("server")

## 1. Load Scan + Auto-Detect Landmarks

In [2]:
SUBJECT_NUMBER = 12
SCANS_FOLDER = "/home/ma7/BA/PG_Subjects11-15"

scan_path = f"{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj"
surface = cedalion.io.read_einstar_obj(scan_path)
print(f"Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces")

# Auto-detect nasion, then all landmarks
from cedalion.geometry.photogrammetry.anonymization.nasion_detector import detect_nasion_auto

nz_point, nz_meta = detect_nasion_auto(surface)
print(f"Auto nasion: {nz_point}, confidence={nz_meta['confidence']:.2f}")

landmarks = detect_landmarks_from_nasion(surface, nz_point)

print(f"\nDetected landmarks:")
for label in landmarks.label.values:
    pos = landmarks.sel(label=label).pint.dequantify().values
    print(f"  {label}: X={pos[0]:7.1f}, Y={pos[1]:7.1f}, Z={pos[2]:7.1f}")

Loaded: 624,121 vertices, 1,169,802 faces
Auto nasion: [108.430481 148.957367 435.152344], confidence=0.48

Detected landmarks:
  Nz: X=  108.4, Y=  149.0, Z=  435.2
  Iz: X=  118.4, Y=  -46.2, Z=  439.6
  Cz: X=  204.9, Y=   42.6, Z=  437.6
  LPA: X=  102.4, Y=   47.4, Z=  532.4
  RPA: X=   80.0, Y=   32.6, Z=  350.6


## 2. Compute Facial Region Mask

In [3]:
facial_landmarks = detect_facial_landmarks(surface, landmarks)

facial_mask = get_facial_region_mask(
    surface=surface,
    facial_landmarks=facial_landmarks,
    protected_points=landmarks,
    protection_radius=15.0 * units.mm,
)

print(f"Facial mask: {facial_mask.sum():,} / {len(facial_mask):,} vertices "
      f"({100 * facial_mask.sum() / len(facial_mask):.1f}%)")

Facial mask: 113,406 / 624,121 vertices (18.2%)


## 3. Delete Facial Vertices

In [4]:
import trimesh

mesh_copy = surface.mesh.copy()
# Remove faces where ANY vertex is in the facial region
face_verts_in_mask = facial_mask[mesh_copy.faces]  # (nfaces, 3) bool
faces_to_remove = face_verts_in_mask.any(axis=1)
mesh_copy.update_faces(~faces_to_remove)
mesh_copy.remove_unreferenced_vertices()
print(f"Original: {surface.nvertices:,} verts, {surface.nfaces:,} faces")
print(f"After deletion: {len(mesh_copy.vertices):,} verts, {len(mesh_copy.faces):,} faces")
print(f"Removed: {surface.nvertices - len(mesh_copy.vertices):,} verts, {surface.nfaces - len(mesh_copy.faces):,} faces")

Original: 624,121 verts, 1,169,802 faces
After deletion: 510,668 verts, 955,712 faces
Removed: 113,453 verts, 214,090 faces


## 4. Visualize Deletion Region

In [6]:
from vtk.util.numpy_support import vtk_to_numpy

vtk_surface = VTKSurface.from_trimeshsurface(surface)
pv_mesh = pv.wrap(vtk_surface.mesh)

# Extract baked vertex colors from VTK (texture already rasterized to per-vertex)
vtk_colors = vtk_surface.mesh.GetPointData().GetScalars()
orig_colors = vtk_to_numpy(vtk_colors).copy()  # (N, 4) RGBA

# Paint facial region red
orig_colors[facial_mask, :3] = [255, 50, 50]
pv_mesh.point_data["mask_colors"] = orig_colors

pvplt = pv.Plotter()
pvplt.add_mesh(pv_mesh, scalars="mask_colors", rgb=True)
pvplt.add_text("Facial Region to Delete (red)", position="upper_left", font_size=14)
pvplt.show()

Widget(value='<iframe src="http://localhost:40875/index.html?ui=P_0x7f897b41c050_1&reconnect=auto" class="pyvi…

## 5. Result: Original vs Anonymized

In [ ]:
from cedalion.vtktutils import trimesh_to_vtk_polydata

# Convert anonymized mesh to VTK (handles TextureVisuals -> vertex colors)
anon_vtk_mesh = trimesh_to_vtk_polydata(mesh_copy)

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, surface, opacity=1.0)
pvplt.add_text("Original", position="upper_left", font_size=14)

pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk_mesh), rgb=True)
pvplt.add_text("Anonymized", position="upper_left", font_size=14)

pvplt.link_views()
pvplt.show()